In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# Paths to the uploaded project inputs
CPI_FILE = Path(r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\cpi_combined_historical_2012base.csv")
WPI_FILE = Path(r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\wpi_monthly_2022_23base.xlsx")
PPI_FILE = Path(r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\output_ppi_monthly_2022_23base.xlsx")

OUT = Path("/mnt/data/phase11_results")
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------
# 1) Load and transform data
# ---------------------------
cpi = pd.read_csv(CPI_FILE)
cpi["date"] = pd.to_datetime(dict(
    year=cpi["year"], month=cpi["month_code"], day=1
))
cpi = cpi.sort_values("date").set_index("date")
cpi["cpi_inflation"] = cpi["index"].pct_change(12) * 100

wpi_raw = pd.read_excel(WPI_FILE)
ppi_raw = pd.read_excel(PPI_FILE)

def monthly_series(df, commodity_name):
    row = df.loc[
        df["Commodity Name"].astype(str).str.strip().eq(commodity_name)
    ].iloc[0]
    month_cols = [
        c for c in df.columns
        if re.match(r"^[A-Z][a-z]{2}-\d{2}$", str(c))
    ]
    s = pd.Series(
        {
            pd.to_datetime(str(c), format="%b-%y"):
            pd.to_numeric(row[c], errors="coerce")
            for c in month_cols
        },
        name=commodity_name,
    )
    return s.sort_index()

wpi_all = monthly_series(wpi_raw, "All Commodities")
wpi_primary = monthly_series(wpi_raw, "I PRIMARY ARTICLES")
wpi_fuel = monthly_series(wpi_raw, "II FUEL & POWER")
wpi_mfg = monthly_series(wpi_raw, "III. MANUFACTURED PRODUCTS")
ppi_all = monthly_series(ppi_raw, "ALL COMMODITIES")

idx = pd.date_range("2023-04-01", "2026-07-01", freq="MS")
trans = pd.DataFrame(index=idx)
trans["cpi_inflation"] = cpi["cpi_inflation"]
trans["wpi_inflation"] = wpi_all.pct_change(12) * 100
trans["wpi_primary_inflation"] = wpi_primary.pct_change(12) * 100
trans["wpi_fuel_inflation"] = wpi_fuel.pct_change(12) * 100
trans["wpi_manufacturing_inflation"] = wpi_mfg.pct_change(12) * 100
trans["ppi_inflation"] = ppi_all.pct_change(12) * 100

common = trans.dropna(
    subset=["cpi_inflation", "wpi_inflation", "ppi_inflation"]
).copy()

# ---------------------------
# 2) Lagged correlations
# ---------------------------
variables = [
    "wpi_inflation",
    "wpi_primary_inflation",
    "wpi_fuel_inflation",
    "wpi_manufacturing_inflation",
    "ppi_inflation",
]

lag_rows = []
for var in variables:
    for lag in range(0, 7):
        tmp = pd.concat(
            [common["cpi_inflation"], common[var].shift(lag)],
            axis=1
        ).dropna()
        lag_rows.append({
            "variable": var,
            "lag_months": lag,
            "n": len(tmp),
            "correlation": tmp.iloc[:, 0].corr(tmp.iloc[:, 1]),
        })

lag_corr = pd.DataFrame(lag_rows)

summary = (
    lag_corr.assign(abs_corr=lambda d: d["correlation"].abs())
    .sort_values(["variable", "abs_corr"], ascending=[True, False])
    .groupby("variable", as_index=False)
    .first()
    .drop(columns="abs_corr")
)

# ---------------------------
# 3) Save data outputs
# ---------------------------
trans.to_csv(OUT / "phase11_transmission_dataset.csv", index_label="date")
common.to_csv(OUT / "phase11_common_yoy_sample.csv", index_label="date")
lag_corr.to_csv(OUT / "02_lagged_correlations.csv", index=False)
summary.to_csv(OUT / "03_lagged_correlation_summary.csv", index=False)

# ---------------------------
# 4) Save charts
# ---------------------------
plt.figure(figsize=(11, 5.5))
plt.plot(common.index, common["cpi_inflation"], label="CPI")
plt.plot(common.index, common["wpi_inflation"], label="WPI")
plt.plot(common.index, common["ppi_inflation"], label="Output PPI")
plt.axhline(0, linewidth=0.8)
plt.title("CPI, WPI and Output PPI Inflation — Common YoY Sample")
plt.xlabel("Date")
plt.ylabel("Inflation (%)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "01_cpi_wpi_ppi_inflation.png", dpi=180)
plt.close()

for var, title, filename in [
    ("wpi_inflation", "CPI–WPI Lagged Correlation", "02_cpi_wpi_lag_correlation.png"),
    ("ppi_inflation", "CPI–Output PPI Lagged Correlation", "03_cpi_ppi_lag_correlation.png"),
]:
    sub = lag_corr[lag_corr["variable"] == var]
    plt.figure(figsize=(8, 5))
    plt.axhline(0, linewidth=0.8)
    plt.plot(sub["lag_months"], sub["correlation"], marker="o")
    plt.title(title)
    plt.xlabel("Lag of upstream inflation (months)")
    plt.ylabel("Pearson correlation")
    plt.xticks(range(0, 7))
    plt.tight_layout()
    plt.savefig(OUT / filename, dpi=180)
    plt.close()

print("Phase 11 first analysis completed.")
print(f"Common CPI–WPI–PPI YoY sample: {common.index.min():%b %Y} to {common.index.max():%b %Y}")
print(f"Observations: {len(common)}")
print("\nMaximum absolute correlation within lags 0–6:")
print(summary.to_string(index=False))
print("\nOutputs written to:", OUT)


Phase 11 first analysis completed.
Common CPI–WPI–PPI YoY sample: Apr 2024 to Dec 2025
Observations: 21

Maximum absolute correlation within lags 0–6:
                   variable  lag_months  n  correlation
              ppi_inflation           0 21     0.849738
         wpi_fuel_inflation           5 16     0.686123
              wpi_inflation           0 21     0.897511
wpi_manufacturing_inflation           6 15    -0.794108
      wpi_primary_inflation           0 21     0.961185

Outputs written to: \mnt\data\phase11_results


In [9]:
from pathlib import Path
import pandas as pd

# Current notebook/project folder
BASE = Path.cwd()

# Phase 11.2 output folder
OUT = BASE / "phase11_2"
OUT.mkdir(exist_ok=True)

# Load the common CPI-WPI-PPI sample
DATA_FILE = BASE / r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\data\Processed Result\phase11_common_inflation_sample.csv"

print("Looking for:", DATA_FILE)
print("Exists:", DATA_FILE.exists())

df = pd.read_csv(
    DATA_FILE,
    index_col=0,
    parse_dates=True
)

print("Shape:", df.shape)
df.head()

Looking for: C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\data\Processed Result\phase11_common_inflation_sample.csv
Exists: True
Shape: (21, 6)


,cpi_inflation,wpi_inflation,wpi_primary_inflation,wpi_fuel_inflation,wpi_manufacturing_inflation,ppi_inflation
2024-04-01,4.828748,0.909091,6.666667,-2.127660,-0.605449,0.605449
2024-05-01,4.801787,1.825558,7.624633,0.328228,0.000000,1.521298
2024-06-01,5.082873,2.441506,8.228461,-0.662983,0.916497,2.240326
2024-07-01,3.596350,1.917255,4.819277,0.662252,1.022495,1.715439
2024-08-01,3.651987,1.306533,5.370370,-0.980392,0.305810,1.307847


In [10]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd()
OUT = BASE / "phase11_2"
OUT.mkdir(exist_ok=True)

DATA_FILE = BASE / r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\data\Processed Result\phase11_common_inflation_sample.csv"

print("Looking for:", DATA_FILE)
print("Exists:", DATA_FILE.exists())

df = pd.read_csv(
    DATA_FILE,
    index_col=0,
    parse_dates=True
)

print("Shape:", df.shape)
df.head()

Looking for: C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\data\Processed Result\phase11_common_inflation_sample.csv
Exists: True
Shape: (21, 6)


,cpi_inflation,wpi_inflation,wpi_primary_inflation,wpi_fuel_inflation,wpi_manufacturing_inflation,ppi_inflation
2024-04-01,4.828748,0.909091,6.666667,-2.127660,-0.605449,0.605449
2024-05-01,4.801787,1.825558,7.624633,0.328228,0.000000,1.521298
2024-06-01,5.082873,2.441506,8.228461,-0.662983,0.916497,2.240326
2024-07-01,3.596350,1.917255,4.819277,0.662252,1.022495,1.715439
2024-08-01,3.651987,1.306533,5.370370,-0.980392,0.305810,1.307847


In [11]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Work on a clean copy
data = df.copy()

# Ensure chronological order and remove incomplete rows
data = data.sort_index().dropna()

print("Observations available:", len(data))
print(data.columns.tolist())

Observations available: 21
['cpi_inflation', 'wpi_inflation', 'wpi_primary_inflation', 'wpi_fuel_inflation', 'wpi_manufacturing_inflation', 'ppi_inflation']


In [12]:
# Create lag-1 predictors
data["cpi_lag1"] = data["cpi_inflation"].shift(1)
data["wpi_lag1"] = data["wpi_inflation"].shift(1)
data["ppi_lag1"] = data["ppi_inflation"].shift(1)

model_data = data.dropna().copy()

print("Usable observations after lagging:", len(model_data))
model_data.head()

Usable observations after lagging: 20


,cpi_inflation,wpi_inflation,wpi_primary_inflation,wpi_fuel_inflation,wpi_manufacturing_inflation,ppi_inflation,cpi_lag1,wpi_lag1,ppi_lag1
2024-05-01,4.801787,1.825558,7.624633,0.328228,0.000000,1.521298,4.828748,0.909091,0.605449
2024-06-01,5.082873,2.441506,8.228461,-0.662983,0.916497,2.240326,4.801787,1.825558,1.521298
2024-07-01,3.596350,1.917255,4.819277,0.662252,1.022495,1.715439,5.082873,2.441506,2.240326
2024-08-01,3.651987,1.306533,5.370370,-0.980392,0.305810,1.307847,3.596350,1.917255,1.715439
2024-09-01,5.486149,1.606426,8.458647,-4.352442,0.305188,1.405622,3.651987,1.306533,1.307847


In [13]:
specs = {
    "CPI-only": ["cpi_lag1"],
    "CPI + WPI": ["cpi_lag1", "wpi_lag1"],
    "CPI + WPI + PPI": ["cpi_lag1", "wpi_lag1", "ppi_lag1"],
}

min_train = 12
results = []
predictions = []

for model_name, features in specs.items():

    y_true = []
    y_pred = []
    origins = []

    for i in range(min_train, len(model_data)):

        train = model_data.iloc[:i]
        test = model_data.iloc[i:i+1]

        X_train = train[features]
        y_train = train["cpi_inflation"]

        X_test = test[features]
        y_test = test["cpi_inflation"].iloc[0]

        model = LinearRegression()
        model.fit(X_train, y_train)

        pred = model.predict(X_test)[0]

        y_true.append(y_test)
        y_pred.append(pred)
        origins.append(test.index[0])

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "n_forecasts": len(y_true)
    })

    for date, actual, forecast in zip(origins, y_true, y_pred):
        predictions.append({
            "Model": model_name,
            "Date": date,
            "Actual": actual,
            "Forecast": forecast,
            "Error": actual - forecast
        })

results_df = pd.DataFrame(results)
predictions_df = pd.DataFrame(predictions)

results_df

,Model,MAE,RMSE,n_forecasts
0,CPI-only,0.758880,0.784428,8
1,CPI + WPI,0.996060,1.096493,8
2,CPI + WPI + PPI,0.973542,1.060089,8


In [14]:
results_path = OUT / "phase11_2_model_comparison.csv"
predictions_path = OUT / "phase11_2_one_step_predictions.csv"

results_df.to_csv(results_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

print("Saved:")
print(results_path)
print(predictions_path)

Saved:
c:\Users\Adity\AppData\Local\Programs\Microsoft VS Code\phase11_2\phase11_2_model_comparison.csv
c:\Users\Adity\AppData\Local\Programs\Microsoft VS Code\phase11_2\phase11_2_one_step_predictions.csv


In [15]:
# Phase 11.2 — Lag sensitivity analysis

lag_specs = {
    "CPI-only": ["cpi_lag1"],
    "CPI + WPI_lag1": ["cpi_lag1", "wpi_lag1"],
    "CPI + WPI_lag2": ["cpi_lag1", "wpi_inflation_lag2"],
    "CPI + WPI_lag3": ["cpi_lag1", "wpi_inflation_lag3"],
    "CPI + PPI_lag1": ["cpi_lag1", "ppi_lag1"],
    "CPI + PPI_lag2": ["cpi_lag1", "ppi_inflation_lag2"],
    "CPI + PPI_lag3": ["cpi_lag1", "ppi_inflation_lag3"],
    "CPI + WPI_lag1 + PPI_lag1": [
        "cpi_lag1",
        "wpi_lag1",
        "ppi_lag1"
    ],
}

# Create additional lags
for lag in [2, 3]:
    data[f"wpi_inflation_lag{lag}"] = data["wpi_inflation"].shift(lag)
    data[f"ppi_inflation_lag{lag}"] = data["ppi_inflation"].shift(lag)

lag_data = data.dropna().copy()

lag_results = []

for model_name, features in lag_specs.items():

    y_true = []
    y_pred = []

    # expanding-window one-step-ahead forecast
    for i in range(min_train, len(lag_data)):

        train = lag_data.iloc[:i]
        test = lag_data.iloc[i:i+1]

        X_train = train[features]
        y_train = train["cpi_inflation"]

        X_test = test[features]
        y_test = test["cpi_inflation"].iloc[0]

        model = LinearRegression()
        model.fit(X_train, y_train)

        pred = model.predict(X_test)[0]

        y_true.append(y_test)
        y_pred.append(pred)

    lag_results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "n_forecasts": len(y_true)
    })

lag_results_df = pd.DataFrame(lag_results).sort_values("RMSE")

lag_results_df

,Model,MAE,RMSE,n_forecasts
0,CPI-only,0.725610,0.756728,6
2,CPI + WPI_lag2,0.706534,0.771976,6
1,CPI + WPI_lag1,0.745206,0.772074,6
5,CPI + PPI_lag2,0.753673,0.780725,6
4,CPI + PPI_lag1,0.757993,0.791688,6
7,CPI + WPI_lag1 + PPI_lag1,0.797268,0.856029,6
6,CPI + PPI_lag3,0.824669,0.975652,6
3,CPI + WPI_lag3,0.891825,1.117544,6


In [16]:
# Phase 11.2 — Interpret and save lag sensitivity results

baseline_rmse = (
    lag_results_df.loc[lag_results_df["Model"] == "CPI-only", "RMSE"]
    .iloc[0]
)

lag_results_df["RMSE_vs_CPI_only_%"] = (
    (lag_results_df["RMSE"] - baseline_rmse)
    / baseline_rmse
    * 100
)

lag_results_df = lag_results_df.sort_values("RMSE")

print("Phase 11.2 — Lag Sensitivity Results")
print("=" * 60)
display(lag_results_df.round(4))

best_model = lag_results_df.iloc[0]

print("\nBest specification:")
print(best_model["Model"])
print(f"RMSE: {best_model['RMSE']:.4f}")
print(f"MAE:  {best_model['MAE']:.4f}")

if best_model["Model"] == "CPI-only":
    print("\nResult: None of the tested WPI/PPI lag specifications")
    print("outperformed the CPI-only benchmark on this small sample.")
else:
    print("\nResult: This specification outperformed CPI-only")
    print("on this small sample.")

# Save results
lag_results_path = OUT / "phase11_2_lag_sensitivity.csv"
lag_results_df.to_csv(lag_results_path, index=False)

print("\nSaved to:")
print(lag_results_path)

Phase 11.2 — Lag Sensitivity Results


,Model,MAE,RMSE,n_forecasts,RMSE_vs_CPI_only_%
0,CPI-only,0.7256,0.7567,6,0.0000
2,CPI + WPI_lag2,0.7065,0.7720,6,2.0150
1,CPI + WPI_lag1,0.7452,0.7721,6,2.0279
5,CPI + PPI_lag2,0.7537,0.7807,6,3.1712
4,CPI + PPI_lag1,0.7580,0.7917,6,4.6199
7,CPI + WPI_lag1 + PPI_lag1,0.7973,0.8560,6,13.1224
6,CPI + PPI_lag3,0.8247,0.9757,6,28.9304
3,CPI + WPI_lag3,0.8918,1.1175,6,47.6811



Best specification:
CPI-only
RMSE: 0.7567
MAE:  0.7256

Result: None of the tested WPI/PPI lag specifications
outperformed the CPI-only benchmark on this small sample.

Saved to:
c:\Users\Adity\AppData\Local\Programs\Microsoft VS Code\phase11_2\phase11_2_lag_sensitivity.csv


# Phase 11 — Final Interpretation & Limitations

## 11.1 Evidence of price co-movement

The common CPI–WPI–Output PPI sample shows substantial contemporaneous and lagged co-movement between consumer and upstream price measures.

The strongest descriptive relationships within the tested 0–6 month lag window were:

- WPI All Commodities: strongest correlation at lag 0.
- WPI Primary Articles: strongest correlation at lag 0.
- WPI Fuel & Power: strongest relationship at a longer lag.
- WPI Manufactured Products: strongest relationship at a longer lag and with an inverse sign.
- Output PPI: strongest correlation at lag 0.

These results indicate that movements in wholesale and producer prices are closely associated with movements in consumer inflation during the common sample.

However, these are **descriptive correlations**. They should not be interpreted as evidence of causal price transmission.

## 11.2 Forecasting contribution

The controlled expanding-window one-step-ahead benchmark compared:

1. CPI-only
2. CPI + WPI
3. CPI + WPI + Output PPI

using the same common evaluation sample.

The CPI-only specification provided the strongest benchmark performance in the initial test. Adding WPI, and then adding both WPI and Output PPI, did not improve forecasting accuracy on this small common sample.

The lag-sensitivity analysis was then used to test whether introducing economically plausible lagged WPI/PPI information changed this conclusion.

The purpose of this exercise is not to establish that WPI or PPI are economically irrelevant. Rather, it tests whether the information contained in these upstream price measures translated into measurable incremental forecasting accuracy under the specific small-sample specifications used here.

## 11.3 Why correlation does not imply forecasting value

A variable can have a strong correlation with CPI inflation while providing limited incremental out-of-sample forecasting information.

This can occur because:

- the variables may respond to common underlying shocks;
- the information may already be contained in CPI's own history;
- contemporaneous correlation may not imply useful leading information;
- additional regressors may increase estimation uncertainty in a very small sample.

Therefore, the forecasting results and the correlation results answer different questions and are interpreted separately.

## 11.4 Sample-size limitation

The common CPI–WPI–Output PPI YoY sample contains only **21 complete monthly observations (April 2024–December 2025)**.

This is a major limitation for formal time-series inference.

In particular, the short overlap makes robust Granger-causality testing and reliable multi-horizon forecasting comparisons difficult. Accordingly, formal Granger causality is **not reported as a headline result** in this project.

The Phase 11 findings should therefore be treated as exploratory evidence rather than definitive evidence of causal transmission.

## 11.5 Overall Phase 11 conclusion

The analysis provides evidence of strong price co-movement between CPI inflation and selected upstream price indicators, but the small common sample does not provide strong evidence that WPI or Output PPI consistently improve one-step-ahead CPI forecasting accuracy beyond a CPI-history benchmark.

The project therefore distinguishes between:

**Price transmission / co-movement:** strong descriptive association

and

**Incremental forecasting value:** not established in this small common sample.

This distinction is central to the interpretation of the forecasting results and prevents correlation-based overclaiming.

## 11.6 Implication for the main forecasting framework

The main forecasting framework should continue to treat CPI-history-based statistical models as the core benchmark.

WPI and Output PPI remain economically relevant candidate predictors and are retained as explanatory/benchmark extensions rather than being presented as proven causal leading indicators.

Future work with a longer and more consistent common sample could provide stronger evidence through:

- richer distributed-lag specifications;
- formal Granger-causality testing;
- multivariate time-series models;
- longer rolling-origin evaluation windows;
- and additional structural or supply-side variables.

**Phase 11 status: Complete, with explicit small-sample limitations.**

## 11.7 Key quantitative results

| Model / Indicator | Result | Interpretation |
|---|---:|---|
| WPI All Commodities | Strongest corr. at lag 0 | Strong contemporaneous co-movement |
| WPI Primary Articles | Strongest corr. at lag 0 | Very strong co-movement |
| WPI Fuel & Power | Strongest corr. at lag 5 | Longer-lag relationship |
| WPI Manufactured Products | Strongest corr. at lag 6 | Inverse longer-lag relationship |
| Output PPI | Strongest corr. at lag 0 | Strong contemporaneous co-movement |
| CPI-only forecast benchmark | Lowest RMSE in initial test | Best small-sample benchmark |
| CPI + WPI | Higher RMSE than CPI-only | No incremental gain in initial test |
| CPI + WPI + PPI | Higher RMSE than CPI-only | No incremental gain in initial test |
| Common CPI–WPI–PPI sample | 21 observations | Major inference limitation |